# 03B — CGAN Hyperparameter Search (Fase 2)

Testa sistematicamente combinações de hiperparâmetros do CGAN.  
Cada configuração é treinada por `GRID_EPOCHS` epochs e avaliada pelo **FID** (Fréchet Inception Distance).  
A melhor configuração é depois re-treinada pelo número total de epochs e o modelo final guardado.

### Estratégia
1. Definir grid de hiperparâmetros
2. Para cada configuração: treinar → calcular FID → registar
3. Selecionar melhor configuração pelo menor FID
4. Re-treinar a melhor configuração com `FULL_EPOCHS` epochs
5. Guardar modelo final e comparar com a Fase 1


In [2]:
import os, sys, json, time, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image as PILImage

import torch
import torch.nn as nn
import torchvision.utils as vutils
import torchvision.transforms as T
import torchvision.models as tv_models
import torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy import linalg

NOTEBOOK_DIR = Path(os.path.abspath("__file__")).parent
SRC_DIR      = (NOTEBOOK_DIR / "../src").resolve()
CKPT_DIR     = (NOTEBOOK_DIR / "../outputs/checkpoints/gan_search").resolve()
SAMPLE_DIR   = (NOTEBOOK_DIR / "../outputs/figures/gan_search").resolve()
PHASE1_CKPT  = (NOTEBOOK_DIR / "../outputs/checkpoints/gan/generator_final.pt").resolve()

sys.path.insert(0, str(SRC_DIR))
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

import kagglehub
dataset_path   = Path(kagglehub.competition_download("aca-tp-2"))
train_dir      = dataset_path / "train"
train_csv_path = dataset_path / "train.csv"

from gan_v2    import Generator, Discriminator, build_cgan, real_label, fake_label
from dataset import ButterflyDataset
from utils   import get_splits, GLOBAL_SEED

device = (
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print(f"Device : {device}")
torch.manual_seed(GLOBAL_SEED)


Device : mps


## Hiperparâmetros

In [3]:
# ── Fixos ────────────────────────────────────────────────────────────────────
NUM_CLASSES   = 75
CLASS_EMB_DIM = 64
BETA1, BETA2  = 0.5, 0.999
BATCH_SIZE    = 64
GRID_EPOCHS   = 25    # epochs por configuração (rápido — só para comparar FID)
FULL_EPOCHS   = 200   # epochs do re-treino da melhor configuração
N_FID_IMGS    = 400   # imagens usadas para calcular FID (por rapidez)
SAVE_EVERY    = 50    # checkpoints durante o re-treino final

# ── Grid ─────────────────────────────────────────────────────────────────────
param_grid = {
    "lr_G":            [2e-4, 1e-4],
    "lr_D":            [1e-4, 5e-5],
    "latent_dim":      [128, 256],
    "n_critic":        [1, 2],          # passos D por passo G
    "label_smoothing": [0.1, 0.0],
}

keys    = list(param_grid.keys())
configs = [
    dict(zip(keys, v))
    for v in itertools.product(*param_grid.values())
    if v[0] >= v[1]      # lr_G >= lr_D (D não deve aprender mais rápido que G)
]
print(f"Configurações a testar: {len(configs)}")
for i, c in enumerate(configs): print(f"  [{i:02d}] {c}")


Configurações a testar: 32
  [00] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 0.1}
  [01] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 0.0}
  [02] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 2, 'label_smoothing': 0.1}
  [03] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 2, 'label_smoothing': 0.0}
  [04] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 256, 'n_critic': 1, 'label_smoothing': 0.1}
  [05] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 256, 'n_critic': 1, 'label_smoothing': 0.0}
  [06] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 256, 'n_critic': 2, 'label_smoothing': 0.1}
  [07] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 256, 'n_critic': 2, 'label_smoothing': 0.0}
  [08] {'lr_G': 0.0002, 'lr_D': 5e-05, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 0.1}
  [09] {'lr_G': 0.0002, 'lr_D': 5e-05, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 

## Dataset

In [4]:
gan_transform = T.Compose([
    T.Resize((64, 64)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

train_csv = pd.read_csv(train_csv_path)
train_df, _ = get_splits(train_csv, seed=GLOBAL_SEED)

with open("class_to_idx.json") as f:
    class_to_idx = json.load(f)
train_df = train_df.copy()
train_df["label_idx"] = train_df["label"].map(class_to_idx)

train_dataset = ButterflyDataset(train_df, img_dir=str(train_dir), transform=gan_transform)
train_loader  = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=(device == "cuda"), drop_last=True,
)
print(f"Treino: {len(train_dataset)} imagens | {len(train_loader)} batches/epoch")


Treino: 4159 imagens | 64 batches/epoch


## Funções de Avaliação (FID)

In [5]:
inception = tv_models.inception_v3(weights="DEFAULT", transform_input=False).to(device)
inception.eval()

def get_inception_features(imgs_01):
    """imgs_01: tensor (N,3,H,W) in [0,1]. Returns numpy (N, 2048)."""
    imgs_299 = F.interpolate(imgs_01.to(device), (299, 299),
                             mode="bilinear", align_corners=False)
    feats = []
    def hook(m, i, o): feats.append(o.squeeze(-1).squeeze(-1).cpu())
    h = inception.avgpool.register_forward_hook(hook)
    with torch.no_grad():
        for i in range(0, len(imgs_299), 64):
            inception(imgs_299[i:i+64])
    h.remove()
    return torch.cat(feats).numpy()

def compute_fid(real_f, fake_f):
    mu1, s1 = real_f.mean(0), np.cov(real_f, rowvar=False)
    mu2, s2 = fake_f.mean(0), np.cov(fake_f, rowvar=False)
    diff    = mu1 - mu2
    covmean, _ = linalg.sqrtm(s1 @ s2, disp=False)
    if np.iscomplexobj(covmean): covmean = covmean.real
    return float(diff @ diff + np.trace(s1 + s2 - 2 * covmean))

# Pré-calcular features das imagens reais (feito uma vez só)
real_sample = train_df.sample(N_FID_IMGS, random_state=GLOBAL_SEED)
real_t = T.Compose([T.Resize((64,64)), T.ToTensor()])
real_imgs = torch.stack([
    real_t(PILImage.open(train_dir / r["filename"]).convert("RGB"))
    for _, r in real_sample.iterrows()
])
real_feats = get_inception_features(real_imgs)
print(f"Features reais pré-calculadas: {real_feats.shape}")


Features reais pré-calculadas: (400, 2048)


## Função de Treino (por configuração)

In [6]:
def train_config(cfg, epochs, loader, device, verbose=True):
    """Treina G e D com cfg durante `epochs` epochs. Devolve (G, D, history)."""
    G_t, D_t = build_cgan(
        latent_dim=cfg["latent_dim"],
        class_emb_dim=CLASS_EMB_DIM,
        num_classes=NUM_CLASSES,
        device=device,
    )
    opt_G = torch.optim.Adam(G_t.parameters(), lr=cfg["lr_G"], betas=(BETA1, BETA2))
    opt_D = torch.optim.Adam(D_t.parameters(), lr=cfg["lr_D"], betas=(BETA1, BETA2))
    crit  = nn.BCEWithLogitsLoss()
    hist  = {"loss_G": [], "loss_D": []}

    for epoch in range(epochs):
        G_t.train(); D_t.train()
        ep_g, ep_d = 0., 0.
        for step, (imgs, labels) in enumerate(loader):
            imgs, labels = imgs.to(device), labels.to(device)
            b = imgs.size(0)

            # ── D ──────────────────────────────────────────────────────────
            opt_D.zero_grad()
            loss_dr = crit(D_t(imgs, labels),
                           real_label(b, device, cfg["label_smoothing"]))
            z  = torch.randn(b, cfg["latent_dim"], device=device)
            fl = torch.randint(0, NUM_CLASSES, (b,), device=device)
            loss_df = crit(D_t(G_t(z, fl).detach(), fl), fake_label(b, device))
            loss_d  = (loss_dr + loss_df) * 0.5
            loss_d.backward(); opt_D.step()

            # ── G (cada n_critic passos) ────────────────────────────────────
            if step % cfg["n_critic"] == 0:
                opt_G.zero_grad()
                z  = torch.randn(b, cfg["latent_dim"], device=device)
                fl = torch.randint(0, NUM_CLASSES, (b,), device=device)
                loss_g = crit(D_t(G_t(z, fl), fl), real_label(b, device, 0.0))
                loss_g.backward(); opt_G.step()

            ep_g += loss_g.item(); ep_d += loss_d.item()

        hist["loss_G"].append(ep_g / len(loader))
        hist["loss_D"].append(ep_d / len(loader))

        if verbose and (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:03d}/{epochs} | "
                  f"loss_G={hist['loss_G'][-1]:.4f}  loss_D={hist['loss_D'][-1]:.4f}")

    return G_t, D_t, hist


def eval_fid(G_model, latent_dim, n_imgs, device):
    """Gera n_imgs e calcula FID contra real_feats pré-calculado."""
    G_model.eval()
    fake_list = []
    with torch.no_grad():
        per_class = max(1, n_imgs // NUM_CLASSES)
        for c in range(NUM_CLASSES):
            lbl = torch.full((per_class,), c, dtype=torch.long, device=device)
            z   = torch.randn(per_class, latent_dim, device=device)
            out = G_model(z, lbl)
            fake_list.append((out * 0.5 + 0.5).clamp(0,1).cpu())
    fake_imgs = torch.cat(fake_list)
    fake_feats = get_inception_features(fake_imgs)
    return compute_fid(real_feats, fake_feats)

print("Funções definidas.")


Funções definidas.


## Grid Search

In [7]:
results = []

for i, cfg in enumerate(configs):
    print(f"\n[{i+1:02d}/{len(configs)}] {cfg}")
    t0 = time.time()

    G_tmp, D_tmp, hist = train_config(cfg, GRID_EPOCHS, train_loader, device, verbose=False)
    fid = eval_fid(G_tmp, cfg["latent_dim"], N_FID_IMGS, device)

    elapsed = time.time() - t0
    results.append({**cfg, "fid": fid, "time_s": elapsed})
    print(f"  → FID = {fid:.2f}  ({elapsed:.0f}s)")

    # Limpar memória
    del G_tmp, D_tmp
    if device == "cuda": torch.cuda.empty_cache()

results_df = pd.DataFrame(results).sort_values("fid").reset_index(drop=True)
results_df.to_csv(CKPT_DIR / "grid_search_results.csv", index=False)
print("\n── Grid Search Concluída ──")
print(results_df.to_string(index=False))



[01/32] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 0.1}
  → FID = 284.53  (859s)

[02/32] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 1, 'label_smoothing': 0.0}
  → FID = 283.91  (780s)

[03/32] {'lr_G': 0.0002, 'lr_D': 0.0001, 'latent_dim': 128, 'n_critic': 2, 'label_smoothing': 0.1}


Python(94369) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94371) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94372) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94374) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94652) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94653) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94654) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94655) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94681) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94682) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(94683) Malloc

KeyboardInterrupt: 

## Visualização dos Resultados

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# FID por configuração
axes[0].barh(range(len(results_df)), results_df["fid"], color="steelblue")
axes[0].set_yticks(range(len(results_df)))
axes[0].set_yticklabels(
    [f"lr_G={r.lr_G:.0e} lr_D={r.lr_D:.0e} z={r.latent_dim} nc={r.n_critic} sm={r.label_smoothing}"
     for _, r in results_df.iterrows()],
    fontsize=7
)
axes[0].set_xlabel("FID (↓ melhor)")
axes[0].set_title("FID por Configuração")
axes[0].axvline(results_df["fid"].iloc[0], color="tomato", linestyle="--", label="Melhor")
axes[0].legend()

# Impacto de cada hiperparâmetro no FID (média por valor)
ax = axes[1]
params_to_plot = ["lr_G", "latent_dim", "n_critic", "label_smoothing"]
means = {p: results_df.groupby(p)["fid"].mean() for p in params_to_plot}
x = np.arange(len(params_to_plot))
width = 0.35
for j, p in enumerate(params_to_plot):
    vals = means[p]
    for k, (val, fid_m) in enumerate(vals.items()):
        ax.bar(j + k * width / len(vals), fid_m, width / len(vals),
               label=f"{p}={val}" if j == 0 else "_")
ax.set_xticks(x + width / 4)
ax.set_xticklabels(params_to_plot, fontsize=9)
ax.set_ylabel("FID médio"); ax.set_title("Impacto de cada Hiperparâmetro")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(SAMPLE_DIR / "grid_search_results.png", dpi=150, bbox_inches="tight")
plt.show()

best = results_df.iloc[0]
print(f"\n✅ Melhor configuração (FID={best.fid:.2f}):")
for k in param_grid.keys():
    print(f"   {k:20s}: {best[k]}")


## Re-treino Completo da Melhor Configuração

Re-treina com `FULL_EPOCHS` e guarda checkpoints periódicos.


In [ ]:
best_cfg = {k: best[k] for k in param_grid.keys()}
# Converter tipos numpy para python (para serialização)
best_cfg["latent_dim"] = int(best_cfg["latent_dim"])
best_cfg["n_critic"]   = int(best_cfg["n_critic"])

print(f"A re-treinar com: {best_cfg}")

G_best, D_best, _ = train_config(
    best_cfg, FULL_EPOCHS, train_loader, device, verbose=True
)

# Guardar modelo final
torch.save(G_best.state_dict(), CKPT_DIR / "generator_best.pt")
torch.save(D_best.state_dict(), CKPT_DIR / "discriminator_best.pt")

with open(CKPT_DIR / "best_config.json", "w") as f:
    json.dump({**best_cfg, "fid_grid": float(best.fid)}, f, indent=2)

print(f"\n✅ Modelo final guardado em {CKPT_DIR}")


## Comparação: Fase 1 vs Melhor Configuração (Fase 2)

In [ ]:
# FID da Fase 2 (melhor config, treino completo)
fid_phase2 = eval_fid(G_best, best_cfg["latent_dim"], N_FID_IMGS, device)
print(f"FID Fase 2 (melhor config, {FULL_EPOCHS} epochs): {fid_phase2:.2f}")

# FID da Fase 1 (para comparar — carrega o generator da fase 1)
try:
    from gan import Generator
    with open(NOTEBOOK_DIR / "../outputs/checkpoints/gan/config.json") as f:
        phase1_cfg = json.load(f)
    G_phase1 = Generator(
        latent_dim=phase1_cfg["latent_dim"],
        class_emb_dim=phase1_cfg["class_emb_dim"],
        num_classes=phase1_cfg["num_classes"],
    ).to(device)
    G_phase1.load_state_dict(torch.load(PHASE1_CKPT, map_location=device))
    fid_phase1 = eval_fid(G_phase1, phase1_cfg["latent_dim"], N_FID_IMGS, device)
    print(f"FID Fase 1 (config fixa, {phase1_cfg['num_epochs']} epochs): {fid_phase1:.2f}")
    compare = True
except FileNotFoundError:
    print("Modelo da Fase 1 não encontrado — corre o notebook 03 primeiro.")
    compare = False

if compare:
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(["Fase 1\n(config fixa)", f"Fase 2\n(melhor config)"],
                  [fid_phase1, fid_phase2],
                  color=["steelblue", "tomato"], width=0.4)
    for bar, val in zip(bars, [fid_phase1, fid_phase2]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", fontsize=11)
    ax.set_ylabel("FID (↓ melhor)")
    ax.set_title("Comparação FID: Fase 1 vs Fase 2")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(SAMPLE_DIR / "fid_phase1_vs_phase2.png", dpi=150, bbox_inches="tight")
    plt.show()


## Amostras Visuais do Melhor Modelo

In [ ]:
G_best.eval()
fixed_z      = torch.randn(NUM_CLASSES, best_cfg["latent_dim"], device=device)
fixed_labels = torch.arange(NUM_CLASSES, device=device)

with torch.no_grad():
    imgs = G_best(fixed_z, fixed_labels)
    imgs = (imgs * 0.5 + 0.5).clamp(0, 1).cpu()

grid = vutils.make_grid(imgs, nrow=15, padding=2)
fig, ax = plt.subplots(figsize=(20, 6))
ax.imshow(grid.permute(1,2,0).numpy())
ax.axis("off")
ax.set_title(f"Melhor CGAN (Fase 2) — 1 amostra por classe  |  FID={fid_phase2:.1f}", fontsize=13)
plt.tight_layout()
plt.savefig(SAMPLE_DIR / "best_model_samples.png", dpi=150, bbox_inches="tight")
plt.show()
